# 🔍 Embedding Models Compared
## text-embedding-3-small vs MiniLM-L6-v2 vs MiniLM-L12-v2 vs BGE-small vs GTE-small

**What you'll learn:**
- Architecture & specs of 5 popular embedding models
- Head-to-head benchmarks: semantic similarity, clustering, retrieval, classification
- Speed, memory, dimensionality & cost trade-offs
- When to pick which model for your use case

**Models benchmarked:**

| Model | Provider | Dims | Type |
|-------|----------|------|------|
| `text-embedding-3-small` | OpenAI (API) | 1536 | Cloud API |
| `all-MiniLM-L6-v2` | Sentence-Transformers | 384 | Local/Open |
| `all-MiniLM-L12-v2` | Sentence-Transformers | 384 | Local/Open |
| `BAAI/bge-small-en-v1.5` | BAAI | 384 | Local/Open |
| `thenlper/gte-small` | Alibaba DAMO | 384 | Local/Open |

---
> ⚠️ The setup cell installs `sentence-transformers` (~2 min on Colab). Local models run on CPU (free tier).

In [ ]:
# ============================================================
# 📦  SETUP & INSTALL
# ============================================================
!pip install openai sentence-transformers scikit-learn numpy rich tqdm scipy -q

import os, time, json, warnings
from getpass import getpass
import numpy as np
from tqdm.notebook import tqdm
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from scipy.stats import spearmanr
from IPython.display import display, Markdown as IPyMd

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API key: ")

from openai import OpenAI
from sentence_transformers import SentenceTransformer

client = OpenAI()
OPENAI_MODEL = "text-embedding-3-small"

print("✅  OpenAI client ready")
print("⏳  Loading local models (first run downloads weights)...\n")

local_models = {
    "MiniLM-L6-v2":  SentenceTransformer("all-MiniLM-L6-v2"),
    "MiniLM-L12-v2": SentenceTransformer("all-MiniLM-L12-v2"),
    "BGE-small":     SentenceTransformer("BAAI/bge-small-en-v1.5"),
    "GTE-small":     SentenceTransformer("thenlper/gte-small"),
}

print("\n✅  All models loaded!")
for name, model in local_models.items():
    dim = model.get_sentence_embedding_dimension()
    print(f"    {name:<18s}  dim={dim}")

In [ ]:
# ============================================================
# 🧰  UNIFIED EMBEDDING INTERFACE
# ============================================================

class EmbedBenchmark:
    """Unified wrapper for benchmarking any embedding model."""
    
    def __init__(self):
        self.local_models = local_models
        self.client = client
    
    def embed(self, texts, model_name):
        """Embed a list of texts and return (embeddings_np, time, dim)."""
        if model_name == "text-embedding-3-small":
            start = time.time()
            all_embs = []
            for i in range(0, len(texts), 100):
                chunk = texts[i:i+100]
                r = self.client.embeddings.create(model=OPENAI_MODEL, input=chunk)
                all_embs.extend([d.embedding for d in r.data])
            elapsed = time.time() - start
            embs = np.array(all_embs)
        else:
            model = self.local_models[model_name]
            start = time.time()
            embs = model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
            elapsed = time.time() - start
        
        return embs, elapsed, embs.shape[1]

bench = EmbedBenchmark()
ALL_MODELS = ["text-embedding-3-small", "MiniLM-L6-v2", "MiniLM-L12-v2", "BGE-small", "GTE-small"]

print("✅  Benchmark harness ready")

---
## 1️⃣  Benchmark A: Semantic Similarity (STS)

The gold standard for embeddings — do cosine-similar vectors actually mean similar sentences?

In [ ]:
# ============================================================
# 📐  SEMANTIC TEXTUAL SIMILARITY (STS) BENCHMARK
# ============================================================

# Curated STS pairs with human-judged similarity (0=unrelated, 1=identical)
sts_pairs = [
    # High similarity
    ("A dog is running through the grass.",      "A dog runs across a green field.",        0.90),
    ("The stock market crashed today.",           "Financial markets saw a major downturn.", 0.88),
    ("She plays the piano every evening.",        "Every night she practices piano.",        0.92),
    ("The cat sat on the windowsill.",            "A feline was perched by the window.",     0.85),
    ("Machine learning requires large datasets.", "ML models need lots of training data.",   0.91),
    # Medium similarity
    ("The weather is nice today.",                "It is a beautiful sunny afternoon.",      0.65),
    ("He went to the store to buy groceries.",    "She drove to the mall for shopping.",     0.50),
    ("Python is a programming language.",         "JavaScript runs in the browser.",         0.45),
    ("The team won the championship.",            "Athletes celebrated their victory.",      0.60),
    # Low similarity
    ("The cat sat on the mat.",                   "The stock market crashed today.",         0.05),
    ("She plays the piano.",                      "Quantum computing uses qubits.",          0.03),
    ("I love eating pizza.",                      "The Eiffel Tower is in Paris.",           0.02),
    ("Dogs are loyal pets.",                      "Interest rates rose sharply.",            0.04),
    ("The sun is a star.",                        "She finished her homework early.",        0.03),
]

texts_a = [p[0] for p in sts_pairs]
texts_b = [p[1] for p in sts_pairs]
human_scores = np.array([p[2] for p in sts_pairs])

print("📐  Semantic Textual Similarity Benchmark\n")
print(f"    {len(sts_pairs)} sentence pairs with human similarity scores\n")

sts_results = {}
for model_name in ALL_MODELS:
    embs_a, t_a, dim = bench.embed(texts_a, model_name)
    embs_b, t_b, _   = bench.embed(texts_b, model_name)
    
    model_sims = np.array([
        float(sk_cosine(embs_a[i:i+1], embs_b[i:i+1])[0, 0])
        for i in range(len(sts_pairs))
    ])
    
    corr, p_val = spearmanr(human_scores, model_sims)
    
    sts_results[model_name] = {
        "spearman": corr,
        "p_value": p_val,
        "mean_abs_error": float(np.mean(np.abs(human_scores - model_sims))),
        "embed_time": t_a + t_b,
        "dim": dim,
    }
    
    bar = "█" * int(corr * 40)
    print(f"  {model_name:<25s}  rho = {corr:.4f}  {bar}  (p={p_val:.2e}, dim={dim})")

best_model = max(sts_results, key=lambda m: sts_results[m]["spearman"])
print(f"\n  🏆  Best STS correlation: {best_model} (rho = {sts_results[best_model]['spearman']:.4f})")

---
## 2️⃣  Benchmark B: Retrieval (Top-K Accuracy)

Given a query, can the model find the most relevant document in a corpus?

In [ ]:
# ============================================================
# 🔎  RETRIEVAL BENCHMARK
# ============================================================

corpus = [
    "Python list comprehensions provide a concise way to create lists based on existing iterables.",
    "The Great Wall of China stretches over 13,000 miles across northern China.",
    "Photosynthesis converts carbon dioxide and water into glucose using sunlight.",
    "RESTful APIs use HTTP methods like GET, POST, PUT, and DELETE for CRUD operations.",
    "The human brain contains approximately 86 billion neurons connected by trillions of synapses.",
    "Docker containers package applications with their dependencies for consistent deployment.",
    "The Mona Lisa was painted by Leonardo da Vinci in the early 16th century.",
    "Gradient boosting combines weak learners sequentially to create a strong predictive model.",
    "The Pacific Ocean is the largest and deepest ocean covering more than 60 million square miles.",
    "OAuth 2.0 is an authorization framework that enables third-party access to user resources.",
    "CRISPR-Cas9 allows precise editing of DNA sequences in living organisms.",
    "Kubernetes orchestrates containerized workloads across clusters of machines.",
    "The speed of light in a vacuum is approximately 299,792 kilometers per second.",
    "MapReduce processes large datasets in parallel across distributed computing clusters.",
    "The Amazon Rainforest produces about 20 percent of the world oxygen supply.",
]

queries = [
    ("How do you create a list from another list in Python?", 0),
    ("What is the process by which plants make food from sunlight?", 2),
    ("How do containers help with software deployment?", 5),
    ("What machine learning method combines multiple weak models?", 7),
    ("How does the gene editing technology work?", 10),
    ("What system manages containers across multiple servers?", 11),
    ("How can third-party apps access user data securely?", 9),
    ("What framework handles big data processing in parallel?", 13),
    ("How fast does light travel?", 12),
    ("What famous painting did da Vinci create?", 6),
]

print("🔎  Retrieval Benchmark\n")
print(f"    Corpus: {len(corpus)} documents  |  Queries: {len(queries)}\n")

retrieval_results = {}
for model_name in ALL_MODELS:
    corpus_embs, t_c, dim = bench.embed(corpus, model_name)
    query_texts = [q[0] for q in queries]
    query_embs, t_q, _ = bench.embed(query_texts, model_name)
    
    top1_correct = 0
    top3_correct = 0
    mrr_sum = 0
    
    for i, (query_text, gt_idx) in enumerate(queries):
        sims = sk_cosine(query_embs[i:i+1], corpus_embs)[0]
        ranked = np.argsort(-sims)
        
        rank_of_gt = int(np.where(ranked == gt_idx)[0][0]) + 1
        mrr_sum += 1.0 / rank_of_gt
        
        if ranked[0] == gt_idx:
            top1_correct += 1
        if gt_idx in ranked[:3]:
            top3_correct += 1
    
    n = len(queries)
    retrieval_results[model_name] = {
        "top1_acc": top1_correct / n,
        "top3_acc": top3_correct / n,
        "mrr": mrr_sum / n,
        "embed_time": t_c + t_q,
    }
    
    print(f"  {model_name:<25s}  Top-1: {top1_correct/n:.0%}  Top-3: {top3_correct/n:.0%}  MRR: {mrr_sum/n:.4f}  time: {t_c+t_q:.3f}s")

best_ret = max(retrieval_results, key=lambda m: retrieval_results[m]["mrr"])
print(f"\n  🏆  Best retrieval: {best_ret} (MRR = {retrieval_results[best_ret]['mrr']:.4f})")

---
## 3️⃣  Benchmark C: Clustering Quality

In [ ]:
# ============================================================
# 🎯  CLUSTERING BENCHMARK
# ============================================================

cluster_data = [
    # Category 0: Programming
    ("Python decorators add functionality to existing functions without modifying them.", 0),
    ("JavaScript async/await simplifies handling asynchronous operations.", 0),
    ("Git branching strategies help teams manage parallel development workflows.", 0),
    ("SQL joins combine rows from two or more tables based on related columns.", 0),
    ("Object-oriented programming uses classes and objects to structure code.", 0),
    ("TypeScript adds static type checking to JavaScript for better tooling.", 0),
    # Category 1: Biology
    ("DNA replication occurs during the S phase of the cell cycle.", 1),
    ("Mitochondria are the powerhouses of the cell, producing ATP.", 1),
    ("Evolution through natural selection favors organisms best adapted to their environment.", 1),
    ("The human immune system uses antibodies to neutralize pathogens.", 1),
    ("Enzymes are biological catalysts that speed up chemical reactions.", 1),
    ("Ribosomes translate mRNA into proteins through the process of translation.", 1),
    # Category 2: Business/Finance
    ("Revenue growth and profit margins are key indicators of company health.", 2),
    ("Diversified investment portfolios reduce risk through asset allocation.", 2),
    ("Supply chain optimization reduces costs and improves delivery times.", 2),
    ("Market capitalization reflects the total value of a company shares.", 2),
    ("Cash flow statements track money moving in and out of a business.", 2),
    ("Mergers and acquisitions can create synergies and increase market share.", 2),
]

cluster_texts  = [d[0] for d in cluster_data]
cluster_labels = [d[1] for d in cluster_data]
n_clusters = 3

print("🎯  Clustering Benchmark\n")
print(f"    {len(cluster_texts)} texts in {n_clusters} categories\n")

cluster_results = {}
for model_name in ALL_MODELS:
    embs, t_emb, dim = bench.embed(cluster_texts, model_name)
    
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    pred_labels = km.fit_predict(embs)
    
    sil_score = silhouette_score(embs, pred_labels)
    ari = adjusted_rand_score(cluster_labels, pred_labels)
    
    cluster_results[model_name] = {
        "silhouette": sil_score,
        "ari": ari,
        "embed_time": t_emb,
    }
    
    sil_bar = "█" * int(max(0, sil_score) * 30)
    ari_bar = "█" * int(max(0, ari) * 30)
    print(f"  {model_name:<25s}  Silhouette: {sil_score:.4f} {sil_bar}")
    print(f"  {'':<25s}  ARI:        {ari:.4f} {ari_bar}")

best_clust = max(cluster_results, key=lambda m: cluster_results[m]["ari"])
print(f"\n  🏆  Best clustering: {best_clust} (ARI = {cluster_results[best_clust]['ari']:.4f})")

---
## 4️⃣  Benchmark D: Classification (Embeddings as Features)

Use embeddings as feature vectors for a downstream logistic regression classifier.

In [ ]:
# ============================================================
# 🏷️  CLASSIFICATION BENCHMARK
# ============================================================

class_data = [
    # Tech (0)
    ("Kubernetes pods restart automatically when health checks fail.", 0),
    ("React hooks replaced class components for state management.", 0),
    ("CI/CD pipelines automate testing and deployment.", 0),
    ("Redis caches frequently accessed data in memory.", 0),
    ("GraphQL lets clients request exactly the data they need.", 0),
    ("Terraform manages infrastructure as code across cloud providers.", 0),
    ("Microservices communicate via REST APIs or message queues.", 0),
    ("Load balancers distribute traffic across multiple servers.", 0),
    # Sports (1)
    ("The quarterback threw a 40-yard touchdown pass in the fourth quarter.", 1),
    ("The marathon runner finished with a personal best time.", 1),
    ("The basketball team won the championship in overtime.", 1),
    ("The tennis player served 15 aces during the match.", 1),
    ("The soccer team advanced to the semifinals after penalty kicks.", 1),
    ("The swimmer broke the world record in the 100m freestyle.", 1),
    ("The boxer won by unanimous decision after 12 rounds.", 1),
    ("The golfer sank a 30-foot putt to win the tournament.", 1),
    # Cooking (2)
    ("Saute the onions in butter until they are golden and translucent.", 2),
    ("Let the bread dough rise for at least an hour in a warm place.", 2),
    ("Deglaze the pan with white wine to create a flavorful sauce.", 2),
    ("Season the steak generously with salt and pepper before grilling.", 2),
    ("Fold the egg whites gently into the batter to keep it fluffy.", 2),
    ("Simmer the tomato sauce on low heat for 45 minutes.", 2),
    ("Cream the butter and sugar together until light and fluffy.", 2),
    ("Roast the vegetables at 425 degrees until caramelized on the edges.", 2),
]

class_texts  = [d[0] for d in class_data]
class_labels = np.array([d[1] for d in class_data])
label_names  = {0: "Tech", 1: "Sports", 2: "Cooking"}

print("🏷️  Classification Benchmark (5-fold cross-validation)\n")
print(f"    {len(class_texts)} texts, {len(label_names)} classes\n")

class_results = {}
for model_name in ALL_MODELS:
    embs, t_emb, dim = bench.embed(class_texts, model_name)
    
    clf = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(clf, embs, class_labels, cv=5, scoring="accuracy")
    
    class_results[model_name] = {
        "mean_acc": float(scores.mean()),
        "std_acc": float(scores.std()),
        "embed_time": t_emb,
        "dim": dim,
    }
    
    bar = "█" * int(scores.mean() * 40)
    print(f"  {model_name:<25s}  Acc: {scores.mean():.1%} +/- {scores.std():.1%}  {bar}  (dim={dim})")

best_cls = max(class_results, key=lambda m: class_results[m]["mean_acc"])
print(f"\n  🏆  Best classification: {best_cls} (Acc = {class_results[best_cls]['mean_acc']:.1%})")

---
## 5️⃣  Speed & Efficiency Benchmark

In [ ]:
# ============================================================
# ⚡  SPEED & THROUGHPUT BENCHMARK
# ============================================================

speed_texts = [f"Document {i}: {text}" for i, text in enumerate(cluster_texts * 5)]

print(f"⚡  Speed Benchmark — {len(speed_texts)} texts\n")

speed_results = {}
for model_name in ALL_MODELS:
    times = []
    for _ in range(3):
        embs, t, dim = bench.embed(speed_texts, model_name)
        times.append(t)
    
    avg_time  = np.mean(times)
    throughput = len(speed_texts) / avg_time
    
    speed_results[model_name] = {
        "avg_time": avg_time,
        "throughput": throughput,
        "dim": dim,
        "mem_per_vec_bytes": dim * 4,
    }
    
    bar = "█" * int(throughput / 50)
    print(f"  {model_name:<25s}  {avg_time:.3f}s  ({throughput:.0f} texts/s)  {bar}")

# Memory comparison
print("\n💾  Memory per 1M vectors:")
for model_name in ALL_MODELS:
    d = speed_results[model_name]
    mem_mb = (d["dim"] * 4 * 1_000_000) / (1024 * 1024)
    bar = "█" * int(mem_mb / 100)
    print(f"  {model_name:<25s}  dim={d['dim']:>5d}  ->  {mem_mb:,.0f} MB  {bar}")

---
## 6️⃣  Cost Analysis

In [ ]:
# ============================================================
# 💰  COST COMPARISON
# ============================================================

print("💰  Cost Comparison for Embedding 1 Million Texts\n")
print("    (Assuming ~30 tokens per text on average)\n")

tokens_per_text = 30
n_texts = 1_000_000
total_tokens = tokens_per_text * n_texts

# OpenAI pricing
openai_cost = (total_tokens / 1000) * 0.000020

local_throughputs = {
    "MiniLM-L6-v2": 2500,
    "MiniLM-L12-v2": 1500,
    "BGE-small": 1800,
    "GTE-small": 1900,
}

print(f"{'Model':<25s} {'Cost':>12s} {'Time (est)':>14s} {'Notes':<30s}")
print("─" * 85)

openai_time = n_texts / 500
print(f"{'text-embedding-3-small':<25s} ${openai_cost:>10.2f} {openai_time/3600:>10.1f} hrs   {'API cost, no infra needed':<30s}")

for model_name, tps in local_throughputs.items():
    est_time = n_texts / tps
    gpu_cost = (est_time / 3600) * 2.0
    print(f"{model_name:<25s} ${gpu_cost:>10.2f} {est_time/3600:>10.1f} hrs   {'$0 on Colab free, $2/hr GPU':<30s}")

print(f"\n📊  At 1M texts, OpenAI API costs ${openai_cost:.2f}")
print(f"    Local models cost $0 on free Colab (but are slower on CPU)")
print(f"    On a cloud GPU (~$2/hr), local models cost ~$0.20-$0.50 total")

---
## 7️⃣  Grand Summary — All Benchmarks Combined

In [ ]:
# ============================================================
# 🏆  GRAND SUMMARY TABLE
# ============================================================

display(IPyMd("### 🏆 Comprehensive Model Comparison"))

print(f"\n{'Model':<25s} {'STS rho':>8s} {'Ret MRR':>9s} {'Clust ARI':>10s} {'Class Acc':>10s} {'Speed':>12s} {'Dim':>6s}")
print("─" * 85)

for model_name in ALL_MODELS:
    sts_rho  = sts_results.get(model_name, {}).get("spearman", 0)
    ret_mrr  = retrieval_results.get(model_name, {}).get("mrr", 0)
    clust_a  = cluster_results.get(model_name, {}).get("ari", 0)
    cls_acc  = class_results.get(model_name, {}).get("mean_acc", 0)
    spd      = speed_results.get(model_name, {}).get("throughput", 0)
    dim      = speed_results.get(model_name, {}).get("dim", 0)
    
    print(f"  {model_name:<23s} {sts_rho:>8.4f} {ret_mrr:>9.4f} {clust_a:>10.4f} {cls_acc:>9.1%} {spd:>8.0f} t/s {dim:>6d}")

categories = {
    "STS Correlation":  (sts_results,       "spearman"),
    "Retrieval MRR":    (retrieval_results,  "mrr"),
    "Clustering ARI":   (cluster_results,    "ari"),
    "Classification":   (class_results,      "mean_acc"),
    "Speed":            (speed_results,      "throughput"),
}

print("\n🥇  Category Winners:")
for cat_name, (results_dict, key) in categories.items():
    winner = max(results_dict, key=lambda m: results_dict[m].get(key, 0))
    val = results_dict[winner][key]
    suffix = " t/s" if key == "throughput" else ""
    print(f"  {cat_name:<20s}  ->  {winner}  ({val:.4f}{suffix})")

---
## 8️⃣  Decision Guide — Which Model to Pick?

| Your Priority | Best Pick | Why |
|---------------|-----------|-----|
| **Easiest setup** (no infra, no GPU) | `text-embedding-3-small` | Just an API call — no downloads, no GPU, scales instantly |
| **Lowest cost at scale** (100K+ docs) | `all-MiniLM-L6-v2` | Free on CPU/Colab, fastest local model, good-enough quality |
| **Best quality (local)** | `BGE-small-en` or `GTE-small` | Often edge out MiniLM on retrieval & STS benchmarks |
| **Privacy / air-gapped** | Any local model | Data never leaves your machine |
| **Balanced (quality + speed)** | `all-MiniLM-L12-v2` | Middle ground — better than L6, faster than large models |
| **Production RAG pipeline** | `text-embedding-3-small` | Reliable, high quality, dimension reduction available, great docs |

### Dimension Reduction Tip

OpenAI's `text-embedding-3-small` supports **native dimension reduction** — you can request 256 or 512 dims instead of 1536:

```python
client.embeddings.create(
    model="text-embedding-3-small",
    input="your text",
    dimensions=512  # reduce from 1536
)
```

In [ ]:
# ============================================================
# 🧪  BONUS: OpenAI Dimension Reduction Test
# ============================================================

dims_to_test = [256, 512, 1024, 1536]
test_query = "How do containers help with software deployment?"
test_docs  = corpus[:10]

print("🧪  OpenAI Dimension Reduction — Retrieval Quality\n")
print(f"    Query: '{test_query}'")
print(f"    Corpus: {len(test_docs)} docs\n")

for dim in dims_to_test:
    q_resp = client.embeddings.create(model=OPENAI_MODEL, input=test_query, dimensions=dim)
    d_resp = client.embeddings.create(model=OPENAI_MODEL, input=test_docs, dimensions=dim)
    
    q_emb = np.array(q_resp.data[0].embedding).reshape(1, -1)
    d_embs = np.array([d.embedding for d in d_resp.data])
    
    sims = sk_cosine(q_emb, d_embs)[0]
    top3 = np.argsort(-sims)[:3]
    
    mem_per_1m = (dim * 4 * 1_000_000) / (1024**2)
    print(f"  dim={dim:>5d}  |  mem/1M vecs: {mem_per_1m:>7.0f} MB  |  Top-1: doc[{top3[0]}] ({sims[top3[0]]:.4f})")
    for rank, idx in enumerate(top3):
        print(f"             rank {rank+1}: [{idx}] {test_docs[idx][:70]}...  (sim={sims[idx]:.4f})")
    print()

print("💡  Notice how even 256-dim preserves the ranking order in most cases!")

---
## ✅  Complete!

You now have empirical, side-by-side data on 5 embedding models across 4 benchmarks + speed + cost.
Use the decision guide above to pick the right model for **your** project. 🎉